# 🤖 Local Chat LLM
**A chatbot using Google Flan-T5 Large — runs on free Google Colab (T4 GPU)**

### Install dependencies first:
```bash
!pip install transformers torch accelerate sentencepiece
```

In [ ]:
# The backbone of all tensor operations and GPU communication
import torch

# T5 has its own special tokenizer and model class — different from the Auto family
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Google's Flan-T5 Large — instruction-tuned, great at Q&A and factual answers
model_name = "google/flan-t5-large"

# Load the T5 tokenizer — it knows exactly how to encode text for the T5 architecture
tokenizer = T5Tokenizer.from_pretrained(model_name)

# Check if a GPU is available — use it if yes, fall back to CPU if not
device = "cuda" if torch.cuda.is_available() else "cpu"

# Tell the user which device they're running on so they know what to expect speed-wise
print(f"🖥️  Using device: {device}")

# Load the model — float16 on GPU halves memory usage, float32 on CPU for stability
model = T5ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,  # Precision based on device
    device_map="auto",   # Let HuggingFace smartly place layers on best available device
)

# Switch to eval mode — turns off dropout so outputs are consistent and deterministic
model.eval()

# This function takes the user's question and the conversation history and returns Bob's reply
def get_reply(user_input, dialog_history):

    # Only use the last 6 lines of history — enough context without overloading the model
    history_text = "\n".join(dialog_history[-6:])

    # Build a clear instruction-style prompt — Flan-T5 responds best to structured instructions
    prompt = f"""You are a helpful and knowledgeable assistant named Bob.
Answer the question clearly and accurately in 2-3 sentences.
Do not repeat yourself.

Conversation so far:
{history_text}

Now answer this question: {user_input}
Bob:"""

    # Tokenize the prompt into tensor format and send it to the right device (GPU/CPU)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",   # Return PyTorch tensors, not lists
        truncation=True,       # Cut off if prompt exceeds max_length
        max_length=512,        # Cap input at 512 tokens to leave room for the reply
    ).to(device)

    # Generate the reply — torch.no_grad() skips gradient tracking to save memory
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],              # The tokenized prompt
            attention_mask=inputs["attention_mask"],    # Tells model which tokens to attend to
            max_new_tokens=150,         # Generate up to 150 new tokens in the reply
            do_sample=True,             # Use sampling so replies feel natural, not robotic
            top_k=50,                   # Only consider the top 50 probable next tokens
            top_p=0.9,                  # Nucleus sampling — keeps diversity without going off-rails
            temperature=0.7,            # Balanced creativity — clear answers, not too random
            repetition_penalty=2.5,     # Strongly discourage repeating the same phrases
            no_repeat_ngram_size=3,     # Never repeat any sequence of 3 words in the reply
            pad_token_id=tokenizer.eos_token_id,   # Use EOS as padding to suppress warnings
        )

    # Decode the generated token IDs back into human-readable text
    reply = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Strip any leading or trailing whitespace before returning the clean reply
    return reply.strip()

# Assign a name to the human user and the AI assistant
my_name = "Kumar"
your_name = "Bob"

# Start with a blank conversation — this list grows with every exchange
dialog = []

# Tell the user the chat is ready and how to exit when done
print(f"🤖 Chatting with {your_name} (Flan-T5 Large). Type 'quit' to exit.\n")

# Keep the conversation alive until the user decides to leave
while True:

    # Wait for the user to type something and remove any accidental spaces
    user_input = input(f"{my_name}: ").strip()

    # If the user types a goodbye word — thank them and exit cleanly
    if user_input.lower() in ["quit", "exit", "bye"]:
        print(f"\n{your_name}: Goodbye! It was great chatting with you. 👋")
        break

    # If the user accidentally hits Enter with nothing typed — remind them kindly
    if not user_input:
        print("⚠️  Please type something!\n")
        continue

    # Log the user's message into the conversation history before generating a reply
    dialog.append(f"{my_name}: {user_input}")

    # Call get_reply() with both the question and full history — context makes answers better
    bot_reply = get_reply(user_input, dialog)

    # Safety net — if the model somehow returns an empty string, give a polite fallback
    if not bot_reply:
        bot_reply = "I'm not sure about that. Could you rephrase the question?"

    # Print Bob's reply on screen for the user to read
    print(f"{your_name}: {bot_reply}\n")

    # Save Bob's reply into history so the next turn has full conversational context
    dialog.append(f"{your_name}: {bot_reply}")